# Pre-processing

In this stage we clean the dataset and create new variables (Feature Engineering) in preparation for model training. We also split the data to avoid data leakage.

P.S. In the first 2 case-studies we shall skip data splitting to focus on cleaning. 

## Case-Study 01: KDD Cup 1999

This is a dataset created by DARPA about network traffic monitoring during simulated hack attempts. The objective is to create a model that can distinguish between normal network traffic and different hack attempts. 

### Data Acquisition

In [1]:
#Load libraries
import pandas as pd
import os.path as osp

In [4]:
# Based from the data dictionary we learn that the attack types and variables names are found in the .names file

with open(osp.join("..", "data", "kddcup99", "kddcup.names"), "r") as f:
    lines = f.readlines()

# Check the lines
lines

['back,buffer_overflow,ftp_write,guess_passwd,imap,ipsweep,land,loadmodule,multihop,neptune,nmap,normal,perl,phf,pod,portsweep,rootkit,satan,smurf,spy,teardrop,warezclient,warezmaster.\n',
 'duration: continuous.\n',
 'protocol_type: symbolic.\n',
 'service: symbolic.\n',
 'flag: symbolic.\n',
 'src_bytes: continuous.\n',
 'dst_bytes: continuous.\n',
 'land: symbolic.\n',
 'wrong_fragment: continuous.\n',
 'urgent: continuous.\n',
 'hot: continuous.\n',
 'num_failed_logins: continuous.\n',
 'logged_in: symbolic.\n',
 'num_compromised: continuous.\n',
 'root_shell: continuous.\n',
 'su_attempted: continuous.\n',
 'num_root: continuous.\n',
 'num_file_creations: continuous.\n',
 'num_shells: continuous.\n',
 'num_access_files: continuous.\n',
 'num_outbound_cmds: continuous.\n',
 'is_host_login: symbolic.\n',
 'is_guest_login: symbolic.\n',
 'count: continuous.\n',
 'srv_count: continuous.\n',
 'serror_rate: continuous.\n',
 'srv_serror_rate: continuous.\n',
 'rerror_rate: continuous.\n'

In [5]:
# Accessing the first and the second element
print(f'first element: {lines[0]}\nsecond element: {lines[1]}')

first element: back,buffer_overflow,ftp_write,guess_passwd,imap,ipsweep,land,loadmodule,multihop,neptune,nmap,normal,perl,phf,pod,portsweep,rootkit,satan,smurf,spy,teardrop,warezclient,warezmaster.

second element: duration: continuous.



In [6]:
lines[1].split(': ')

['duration', 'continuous.\n']

In [7]:
lines[1].split(': ')[1].strip().replace('.','')

'continuous'

In [8]:
# Maintain an index of all columns, symbolic and continuous
columns = []
symbolic = []
continuous = []

# Looping through every line from the second onwards
for line in lines[1:]:
    line_parts = line.split(': ')
    column_name = line_parts[0]
    column_type = line_parts[1].strip().replace('.','')

    if column_type == 'symbolic':
        symbolic.append(column_name)
    else:
        continuous.append(column_name)

    columns.append(column_name)

# Specify the target variable since this is not in the .names file
target = 'class'
columns.append(target)

# Preview list of columns
columns

['duration',
 'protocol_type',
 'service',
 'flag',
 'src_bytes',
 'dst_bytes',
 'land',
 'wrong_fragment',
 'urgent',
 'hot',
 'num_failed_logins',
 'logged_in',
 'num_compromised',
 'root_shell',
 'su_attempted',
 'num_root',
 'num_file_creations',
 'num_shells',
 'num_access_files',
 'num_outbound_cmds',
 'is_host_login',
 'is_guest_login',
 'count',
 'srv_count',
 'serror_rate',
 'srv_serror_rate',
 'rerror_rate',
 'srv_rerror_rate',
 'same_srv_rate',
 'diff_srv_rate',
 'srv_diff_host_rate',
 'dst_host_count',
 'dst_host_srv_count',
 'dst_host_same_srv_rate',
 'dst_host_diff_srv_rate',
 'dst_host_same_src_port_rate',
 'dst_host_srv_diff_host_rate',
 'dst_host_serror_rate',
 'dst_host_srv_serror_rate',
 'dst_host_rerror_rate',
 'dst_host_srv_rerror_rate',
 'class']

In [9]:
# Accessing the data file
df = pd.read_csv(osp.join('..', 'data', 'kddcup99', 'kddcup.data.corrected'), header=None, names=columns)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4898431 entries, 0 to 4898430
Data columns (total 42 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   duration                     int64  
 1   protocol_type                str    
 2   service                      str    
 3   flag                         str    
 4   src_bytes                    int64  
 5   dst_bytes                    int64  
 6   land                         int64  
 7   wrong_fragment               int64  
 8   urgent                       int64  
 9   hot                          int64  
 10  num_failed_logins            int64  
 11  logged_in                    int64  
 12  num_compromised              int64  
 13  root_shell                   int64  
 14  su_attempted                 int64  
 15  num_root                     int64  
 16  num_file_creations           int64  
 17  num_shells                   int64  
 18  num_access_files             int64  
 19  num_outboun

In [10]:
df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,http,SF,215,45076,0,0,0,0,...,0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,normal.
1,0,tcp,http,SF,162,4528,0,0,0,0,...,1,1.0,0.0,1.00,0.0,0.0,0.0,0.0,0.0,normal.
2,0,tcp,http,SF,236,1228,0,0,0,0,...,2,1.0,0.0,0.50,0.0,0.0,0.0,0.0,0.0,normal.
3,0,tcp,http,SF,233,2032,0,0,0,0,...,3,1.0,0.0,0.33,0.0,0.0,0.0,0.0,0.0,normal.
4,0,tcp,http,SF,239,486,0,0,0,0,...,4,1.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,normal.


In [12]:
# P.S. Since this dataset is very wide, by default pandas would not show all columns
pd.set_option('display.max_columns', None)

df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,http,SF,215,45076,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,normal.
1,0,tcp,http,SF,162,4528,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,2,2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1,1,1.0,0.0,1.00,0.0,0.0,0.0,0.0,0.0,normal.
2,0,tcp,http,SF,236,1228,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2,2,1.0,0.0,0.50,0.0,0.0,0.0,0.0,0.0,normal.
3,0,tcp,http,SF,233,2032,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,2,2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,3,3,1.0,0.0,0.33,0.0,0.0,0.0,0.0,0.0,normal.
4,0,tcp,http,SF,239,486,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,3,3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4,4,1.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,normal.


In [13]:
df.tail()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
4898426,0,tcp,http,SF,212,2288,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,3,32,0.0,0.0,0.0,0.0,1.0,0.0,0.16,3,255,1.0,0.0,0.33,0.05,0.0,0.01,0.0,0.0,normal.
4898427,0,tcp,http,SF,219,236,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,4,33,0.0,0.0,0.0,0.0,1.0,0.0,0.15,4,255,1.0,0.0,0.25,0.05,0.0,0.01,0.0,0.0,normal.
4898428,0,tcp,http,SF,218,3610,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5,34,0.0,0.0,0.0,0.0,1.0,0.0,0.15,5,255,1.0,0.0,0.20,0.05,0.0,0.01,0.0,0.0,normal.
4898429,0,tcp,http,SF,219,1234,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,6,35,0.0,0.0,0.0,0.0,1.0,0.0,0.14,6,255,1.0,0.0,0.17,0.05,0.0,0.01,0.0,0.0,normal.
4898430,0,tcp,http,SF,219,1098,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.0,0.0,0.0,1.0,0.0,0.00,7,255,1.0,0.0,0.14,0.05,0.0,0.01,0.0,0.0,normal.


From this stage we establish a number of facts:
1. This is a supervised machine learning problem, with a labelled dataset. The label is the last column `class`, which needs to be predicted. 
2. This is a large dataset with a number of issues. One would need to read supporting material to have understood the issues to loading the dataset. These were provided for ease of use and academic purposes. 
3. The target variable has a period symbol which needs to be cleaned. 
4. We observe that there is a wide range of different values. 
5. At 1.5GB of memory this is a large dataset and therefore one must be mindful of execution of this notebook.

### Exploratory Data Analysis

In [14]:
# Statistival Analysis
df[continuous].describe()

,duration,src_bytes,dst_bytes,wrong_fragment,urgent,hot,num_failed_logins,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
count,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4898431.0,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06,4.898431e+06
mean,4.834243e+01,1.834621e+03,1.093623e+03,6.487792e-04,7.961733e-06,1.243766e-02,3.205108e-05,8.088304e-03,6.818510e-05,3.674646e-05,1.293496e-02,1.188748e-03,7.430951e-05,1.021143e-03,0.0,3.349734e+02,2.952671e+02,1.779703e-01,1.780370e-01,5.766509e-02,5.773010e-02,7.898842e-01,2.117961e-02,2.826080e-02,2.329811e+02,1.892142e+02,7.537132e-01,3.071111e-02,6.050520e-01,6.464107e-03,1.780911e-01,1.778859e-01,5.792780e-02,5.765941e-02
std,7.233298e+02,9.414311e+05,6.450123e+05,4.285434e-02,7.215084e-03,4.689782e-01,7.299408e-03,3.856481e+00,8.257146e-03,8.082432e-03,3.938075e+00,1.241857e-01,8.737590e-03,3.551048e-02,0.0,2.119908e+02,2.459927e+02,3.818756e-01,3.822541e-01,2.322529e-01,2.326604e-01,3.892958e-01,8.271458e-02,1.405596e-01,6.402094e+01,1.059128e+02,4.111860e-01,1.085432e-01,4.809877e-01,4.125978e-02,3.818382e-01,3.821774e-01,2.309428e-01,2.309777e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,4.500000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,1.210000e+02,1.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,2.550000e+02,4.900000e+01,4.100000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,5.200000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,5.100000e+02,5.100000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,2.550000e+02,2.550000e+02,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,1.032000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,5.110000e+02,5.110000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,2.550000e+02,2.550000e+02,1.000000e+00,4.000000e-02,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,5.832900e+04,1.379964e+09,1.309937e+09,3.000000e+00,1.400000e+01,7.700000e+01,5.000000e+00,7.479000e+03,1.000000e+00,2.000000e+00,7.468000e+03,4.300000e+01,2.000000e+00,9.000000e+00,0.0,5.110000e+02,5.110000e+02,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,2.550000e+02,2.550000e+02,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00

From this statistical overview we have identified:
1. `num_outbound_cmds` is a column entirely made up of 0. Therefore is an irrelevant variable and needs to be removed.
2. There is a wide range of different variables. Training a neural network would be an issue. 

In [15]:
# Checking the range of values of categorical variables
for feature in symbolic:
    print(f'{feature}: {df[feature].unique()}')

protocol_type: <StringArray>
['tcp', 'udp', 'icmp']
Length: 3, dtype: str
service: <StringArray>
[       'http',        'smtp',    'domain_u',        'auth',      'finger',
      'telnet',       'eco_i',         'ftp',       'ntp_u',       'ecr_i',
       'other',       'urp_i',     'private',       'pop_3',    'ftp_data',
     'netstat',     'daytime',         'ssh',        'echo',        'time',
        'name',       'whois',      'domain',         'mtp',      'gopher',
  'remote_job',         'rje',         'ctf',      'supdup',        'link',
      'systat',     'discard',         'X11',       'shell',       'login',
       'imap4',        'nntp',        'uucp',     'pm_dump',         'IRC',
      'Z39_50', 'netbios_dgm',        'ldap',      'sunrpc',     'courier',
        'exec',         'bgp',    'csnet_ns',    'http_443',      'klogin',
     'printer', 'netbios_ssn',       'pop_2',        'nnsp',         'efs',
   'hostnames',   'uucp_path',     'sql_net',       'vmnet',    'is

From this analysis of categorical/symbolic variables we observe:
1. all variables have multiple unique values so are all relevant
2. `land`, `logged_in`, `is_host_login` are boolean and therefore need to be changed accordingly. 

In [16]:
# Checking the target variable
df[target].value_counts()

class
smurf.              2807886
neptune.            1072017
normal.              972781
satan.                15892
ipsweep.              12481
portsweep.            10413
nmap.                  2316
back.                  2203
warezclient.           1020
teardrop.               979
pod.                    264
guess_passwd.            53
buffer_overflow.         30
land.                    21
warezmaster.             20
imap.                    12
rootkit.                 10
loadmodule.               9
ftp_write.                8
multihop.                 7
phf.                      4
perl.                     3
spy.                      2
Name: count, dtype: int64

In [17]:
df[target].value_counts(normalize=True)

class
smurf.              5.732215e-01
neptune.            2.188491e-01
normal.             1.985903e-01
satan.              3.244304e-03
ipsweep.            2.547959e-03
portsweep.          2.125783e-03
nmap.               4.728045e-04
back.               4.497358e-04
warezclient.        2.082299e-04
teardrop.           1.998599e-04
pod.                5.389481e-05
guess_passwd.       1.081979e-05
buffer_overflow.    6.124410e-06
land.               4.287087e-06
warezmaster.        4.082940e-06
imap.               2.449764e-06
rootkit.            2.041470e-06
loadmodule.         1.837323e-06
ftp_write.          1.633176e-06
multihop.           1.429029e-06
phf.                8.165880e-07
perl.               6.124410e-07
spy.                4.082940e-07
Name: proportion, dtype: float64

From the above analysis we observe that:
1. the values of the target variable all have a trailing '.' which needs cleaning.
2. this dataset is heavily inbalanced, with some attack types being represented in single or double digits. 

In [18]:
# Check for missing values
df.isna().sum()

duration                       0
protocol_type                  0
service                        0
flag                           0
src_bytes                      0
dst_bytes                      0
land                           0
wrong_fragment                 0
urgent                         0
hot                            0
num_failed_logins              0
logged_in                      0
num_compromised                0
root_shell                     0
su_attempted                   0
num_root                       0
num_file_creations             0
num_shells                     0
num_access_files               0
num_outbound_cmds              0
is_host_login                  0
is_guest_login                 0
count                          0
srv_count                      0
serror_rate                    0
srv_serror_rate                0
rerror_rate                    0
srv_rerror_rate                0
same_srv_rate                  0
diff_srv_rate                  0
srv_diff_h

In [19]:
df.duplicated().sum()

np.int64(3823439)

From the above analysis we observe:
1. there are no missing values
2. there are 3.8 million duplicate rows

### Data Pre-Processing

Based from the EDA we shall:
1. drop `num_outbound_cmd`
2. normalize numeric variables
3. change categorical variable data types
4. change boolean variable data types
5. clean target variable values
6. remove duplicates
7. engineer a new binary target variable
8. engineer a new 5 class target variable

## Case-Study 02: 

## Case-Study 03: 